# ✅ Solutions — Session 3: Cleaning Data

Worked answers to every exercise plus the **Messy CSV Cleanup** mini project.
Each solution is self-contained and uses pandas 3.x idioms only.

In [1]:
import io

import numpy as np
import pandas as pd

messy = pd.read_csv(
    io.StringIO(
        """name,city,price,units,joined
  ana ,berlin,12.5,3,2023-01-05
BO,Paris,7.5,,2023-02-10
Cara,lisbon,n/a,5,not-a-date
dan,BERLIN,22.0,2,2023-03-15
ana,berlin,12.5,3,2023-01-05
Eve,,15.0,4,2023-04-01
"""
    )
)
messy

,name,city,price,units,joined
0,ana,berlin,12.5,3.0,2023-01-05
1,BO,Paris,7.5,NaN,2023-02-10
2,Cara,lisbon,NaN,5.0,not-a-date
3,dan,BERLIN,22.0,2.0,2023-03-15
4,ana,berlin,12.5,3.0,2023-01-05
5,Eve,NaN,15.0,4.0,2023-04-01


## Exercise 1 — Missing values and unexpected dtypes

**(Easy)** `price` is text (`str`) because it contains `"n/a"`, and `joined` is text because of
`"not-a-date"`. `units` (missing once) and `joined` contain missing values.

In [2]:
print(messy.isna().sum())
print()
print(messy.dtypes)

name      0
city      1
price     1
units     1
joined    0
dtype: int64

name          str
city          str
price     float64
units     float64
joined        str
dtype: object


## Exercise 2 — Normalize text with `.str`

**(Easy)** `strip` removes padding spaces; `title` normalizes the casing.

In [3]:
text_clean = messy.copy()
text_clean["name"] = text_clean["name"].str.strip().str.title()
text_clean["city"] = text_clean["city"].str.strip().str.title()
text_clean[["name", "city"]]

,name,city
0,Ana,Berlin
1,Bo,Paris
2,Cara,Lisbon
3,Dan,Berlin
4,Ana,Berlin
5,Eve,NaN


## Exercise 3 — Convert dtypes, then fill

**(Medium, coding)** `to_numeric`/`to_datetime` with `errors="coerce"` turn bad entries into `NaN`;
the mean fill is computed after coercion, so `"n/a"` never breaks the arithmetic.

In [4]:
converted = text_clean.copy()
converted["price"] = pd.to_numeric(converted["price"], errors="coerce")
converted["joined"] = pd.to_datetime(converted["joined"], errors="coerce")
converted["price"] = converted["price"].fillna(converted["price"].mean())
converted["units"] = converted["units"].fillna(0)
converted[["name", "price", "units", "joined"]]

,name,price,units,joined
0,Ana,12.5,3.0,2023-01-05
1,Bo,7.5,0.0,2023-02-10
2,Cara,13.9,5.0,NaT
3,Dan,22.0,2.0,2023-03-15
4,Ana,12.5,3.0,2023-01-05
5,Eve,15.0,4.0,2023-04-01


## Exercise 4 — Deduplicate and cast `city` to categorical

**(Medium)** `drop_duplicates` removes the repeated `ana` row before the category cast.

In [5]:
print("duplicates before:", int(converted.duplicated().sum()))

final = converted.drop_duplicates().copy()
final["city"] = final["city"].astype("category")

print("duplicates after :", int(final.duplicated().sum()))
print("city dtype       :", final["city"].dtype)
final

duplicates before: 1
duplicates after : 0
city dtype       : category


,name,city,price,units,joined
0,Ana,Berlin,12.5,3.0,2023-01-05
1,Bo,Paris,7.5,0.0,2023-02-10
2,Cara,Lisbon,13.9,5.0,NaT
3,Dan,Berlin,22.0,2.0,2023-03-15
5,Eve,NaN,15.0,4.0,2023-04-01


## Exercise 5 — `astype` vs `to_numeric`

**(Advanced, conceptual)** `astype("float64")` is a strict cast: one unparseable entry (like `"n/a"`) raises
`ValueError` and stops the pipeline. `pd.to_numeric(..., errors="coerce")` converts
unparseable entries to `NaN`, letting you see and handle them. The silent-corruption
trap is the opposite direction: using `astype` on a column that *happens* to look
clean but has a stray value — or dropping the errors with `errors="ignore"` — can
leave text in a numeric column that later sorts lexicographically ("10" < "9").

In [6]:
print("to_numeric is visible and safe:")
print(pd.to_numeric(pd.Series(["1", "n/a", "3"]), errors="coerce").tolist())

to_numeric is visible and safe:
[1.0, nan, 3.0]


## 🚀 Mini Project — Messy CSV Cleanup

In [7]:
raw = pd.read_csv(
    io.StringIO(
        """name,city,signup,spend,visits
  Ana ,berlin,2023-01-05,12.50,3
BO,Paris,2023-02-10,7.5,
Cara,lisbon,not-a-date,n/a,5
dan,BERLIN,2023-03-15,22.00,2
Ana,berlin,2023-01-05,12.50,3
Eve,,2023-04-01,15.0,4
"""
    )
)

# Step 2 — rename columns.
df = raw.rename(columns={"signup": "joined", "spend": "amount"})

# Step 3 — clean text columns and fill empty city.
df["name"] = df["name"].str.strip().str.title()
df["city"] = df["city"].str.strip().str.title().replace({"": np.nan})
df["city"] = df["city"].fillna("Unknown")

# Step 4 — parse dates and numbers safely.
df["joined"] = pd.to_datetime(df["joined"], errors="coerce")
df["amount"] = pd.to_numeric(df["amount"], errors="coerce")

# Step 5 — report, then fill missing values.
print("missing before fill:")
print(df.isna().sum())
df["visits"] = df["visits"].fillna(0).astype("int64")
df["amount"] = df["amount"].fillna(df["amount"].mean())

missing before fill:
name      0
city      0
joined    1
amount    1
visits    1
dtype: int64


In [8]:
# Step 6 — deduplicate on the composite key.
print("duplicates keyed on name+joined:",
      int(df.duplicated(subset=["name", "joined"]).sum()))
df = df.drop_duplicates(subset=["name", "joined"], keep="first")

# Step 7 — category dtype and final report.
df["city"] = df["city"].astype("category")
print("\ndtypes:")
print(df.dtypes)
print("\nclean frame:")
df

duplicates keyed on name+joined: 1

dtypes:
name                 str
city            category
joined    datetime64[us]
amount           float64
visits             int64
dtype: object

clean frame:


,name,city,joined,amount,visits
0,Ana,Berlin,2023-01-05,12.5,3
1,Bo,Paris,2023-02-10,7.5,0
2,Cara,Lisbon,NaT,13.9,5
3,Dan,Berlin,2023-03-15,22.0,2
5,Eve,Unknown,2023-04-01,15.0,4
